# Credit Card Fraud Detection

In [2]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "4"

import random
import warnings
import lightgbm as lgb
import numpy as np
import optuna
import pandas as pd
from sklearn.ensemble import HistGradientBoostingClassifier, RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_recall_curve,
)
from xgboost import XGBClassifier

warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

# ---------------------------------------------------------
# GLOBAL RANDOM SEED FOR FULL REPRODUCIBILITY
# ---------------------------------------------------------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# =========================================================
# STAGE 1: LOAD & CHRONOLOGICAL SPLIT
# =========================================================
DATA_PATH = "610_ps4_training/trainingT1FD/cct_train.csv"
df = pd.read_csv(DATA_PATH)

# Parse timestamps & sort chronologically
df["datetime"] = pd.to_datetime(
    df["trans_date"].astype(str) + " " + df["trans_time"].astype(str)
)
df = df.sort_values("unix_time").reset_index(drop=True)
df["date"] = df["datetime"].dt.date
df["hour"] = df["datetime"].dt.hour
df["day_of_week"] = df["datetime"].dt.day_name()

# Cardholder age
df["dob"] = pd.to_datetime(df["dob"])
df["age"] = ((df["datetime"] - df["dob"]).dt.days / 365.25).astype(int)


# Haversine distance (miles between cardholder home and merchant)
def haversine_np(lat1, lon1, lat2, lon2):
    R = 3958.8
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = (
        np.sin(dphi / 2.0) ** 2
        + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda / 2.0) ** 2
    )
    return 2 * R * np.arctan2(np.sqrt(a), np.sqrt(1 - a))


df["dist_miles"] = haversine_np(
    df["lat"], df["long"], df["merch_lat"], df["merch_long"]
)

# Velocity: minutes since previous transaction on the same credit card
df["mins_since_last_tx"] = df.groupby("cc_num")["unix_time"].diff() / 60.0

# Clean labels & Executive Summary Metrics
df["Status"] = df["is_fraud"].map({0: "Legitimate", 1: "Fraud"})
palette = {"Legitimate": "#2B5B84", "Fraud": "#D9381E"}

total_tx = len(df)
total_vol = df["amt"].sum()
fraud_df = df[df["is_fraud"] == 1]
legit_df = df[df["is_fraud"] == 0]

print("=" * 60)
print("EXECUTIVE SUMMARY METRICS")
print("=" * 60)
print(f"Total Transactions : {total_tx:,}")
print(f"Total Volume ($)   : ${total_vol:,.2f}")
print(f"Legitimate Tx      : {len(legit_df):,} ({len(legit_df)/total_tx*100:.2f}%)")
print(f"Fraud Tx           : {len(fraud_df):,} ({len(fraud_df)/total_tx*100:.4f}%)")
print("=" * 60)

# =========================================================
# STAGE 2: FEATURE ENGINEERING (LEAKAGE-FREE)
# =========================================================
# Fill initial card velocity NaN with a large default (10 days in minutes)
df["mins_since_last_tx"] = df["mins_since_last_tx"].fillna(10 * 24 * 60.0)
df["is_night"] = ((df["hour"] >= 22) | (df["hour"] <= 3)).astype(int)
df["day_of_week_num"] = df["datetime"].dt.dayofweek
df["gender_male"] = (df["gender"] == "M").astype(int)
df["log_amt"] = np.log1p(df["amt"])
df["log_city_pop"] = np.log1p(df["city_pop"])

# One-hot encode category safely
cat_dummies = pd.get_dummies(df["category"], prefix="cat", dtype=int)
df = pd.concat([df, cat_dummies], axis=1)

FEATURE_COLS = [
    "amt",
    "log_amt",
    "hour",
    "is_night",
    "day_of_week_num",
    "age",
    "gender_male",
    "log_city_pop",
    "dist_miles",
    "mins_since_last_tx",
] + list(cat_dummies.columns)

BASELINE_COLS = ["amt", "hour"]

# Chronological Split:
# - Pre-Sept 2022 -> Development (75% AutoML Train / 25% Threshold Tuning Val)
# - Sept 1, 2022 onward -> Final Held-Out Evaluation Test Set
if (df["trans_date"] >= "2022-09-01").sum() > 0:
    dev_df = df[df["trans_date"] < "2022-09-01"].reset_index(drop=True)
    test_df = df[df["trans_date"] >= "2022-09-01"].reset_index(drop=True)
else:
    split_test = int(len(df) * 0.80)
    dev_df = df.iloc[:split_test].reset_index(drop=True)
    test_df = df.iloc[split_test:].reset_index(drop=True)

split_val = int(len(dev_df) * 0.75)
train_df = dev_df.iloc[:split_val]
val_df = dev_df.iloc[split_val:]

X_train, y_train = train_df[FEATURE_COLS], train_df["is_fraud"].astype(int).values
X_val, y_val = val_df[FEATURE_COLS], val_df["is_fraud"].astype(int).values
X_dev, y_dev = dev_df[FEATURE_COLS], dev_df["is_fraud"].astype(int).values
X_test, y_test = test_df[FEATURE_COLS], test_df["is_fraud"].astype(int).values

# Balanced class weight ratio for XGBoost
balanced_scale_pos = (y_train == 0).sum() / max((y_train == 1).sum(), 1)


def find_best_threshold(y_true, y_probs):
    """Finds the probability cutoff that maximizes Fraud F1 (pos_label=1)."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = np.argmax(f1_scores[:-1])
    return float(thresholds[best_idx]), float(f1_scores[best_idx])


# =========================================================
# STAGE 3 & 4: BASELINE + AUTOML GRADIENT BOOSTING SEARCH
# =========================================================
def build_candidate_model(trial):
    family = trial.suggest_categorical(
        "model_family", ["XGBoost", "LightGBM", "HistGradientBoosting"]
    )

    if family == "XGBoost":
        return XGBClassifier(
            n_estimators=trial.suggest_int("xgb_n_estimators", 100, 350),
            learning_rate=trial.suggest_float("xgb_lr", 0.03, 0.2, log=True),
            max_leaves=trial.suggest_int("xgb_max_leaves", 12, 48),
            grow_policy="lossguide",
            max_depth=0,
            subsample=trial.suggest_float("xgb_subsample", 0.7, 1.0),
            colsample_bytree=trial.suggest_float("xgb_colsample", 0.7, 1.0),
            scale_pos_weight=balanced_scale_pos,
            tree_method="hist",
            random_state=RANDOM_SEED,
            n_jobs=4,
        )

    elif family == "LightGBM":
        return lgb.LGBMClassifier(
            n_estimators=trial.suggest_int("lgb_n_estimators", 100, 350),
            learning_rate=trial.suggest_float("lgb_lr", 0.03, 0.2, log=True),
            num_leaves=trial.suggest_int("lgb_num_leaves", 12, 48),
            subsample=trial.suggest_float("lgb_subsample", 0.7, 1.0),
            colsample_bytree=trial.suggest_float("lgb_colsample", 0.7, 1.0),
            class_weight="balanced",
            random_state=RANDOM_SEED,
            n_jobs=4,
            verbose=-1,
        )

    else:  # HistGradientBoosting
        return HistGradientBoostingClassifier(
            max_iter=trial.suggest_int("hgb_max_iter", 100, 350),
            learning_rate=trial.suggest_float("hgb_lr", 0.03, 0.2, log=True),
            max_leaf_nodes=trial.suggest_int("hgb_max_leaves", 12, 48),
            class_weight="balanced",
            random_state=RANDOM_SEED,
        )


def objective(trial):
    clf = build_candidate_model(trial)
    clf.fit(X_train, y_train)
    val_probs = clf.predict_proba(X_val)[:, 1]
    best_thresh, best_val_f1 = find_best_threshold(y_val, val_probs)
    trial.set_user_attr("best_threshold", best_thresh)
    return best_val_f1


print("\nRunning AutoML search across XGBoost, LightGBM & HistGradientBoosting...")
sampler = optuna.samplers.TPESampler(seed=RANDOM_SEED)
study = optuna.create_study(direction="maximize", sampler=sampler)
study.optimize(objective, n_trials=15)  # Increase n_trials (e.g. 30) for deeper search

# Leaderboard of explored AutoML trials
trials_df = study.trials_dataframe()
leaderboard = (
    trials_df[["number", "params_model_family", "value", "user_attrs_best_threshold"]]
    .rename(
        columns={
            "number": "Trial",
            "params_model_family": "Model_Family",
            "value": "Val_Fraud_F1",
            "user_attrs_best_threshold": "Tuned_Threshold",
        }
    )
    .sort_values("Val_Fraud_F1", ascending=False)
)
print("\nTOP AUTOML CANDIDATES (VALIDATION LEADERBOARD):")
print(leaderboard.head(8).to_string(index=False))

# Refit the winning AutoML model on all pre-September development data
best_trial = study.best_trial
best_threshold = best_trial.user_attrs["best_threshold"]
best_model = build_candidate_model(best_trial)

# Update scale_pos_weight if XGBoost won, using full dev set ratio
if best_trial.params["model_family"] == "XGBoost":
    best_model.set_params(
        scale_pos_weight=(y_dev == 0).sum() / max((y_dev == 1).sum(), 1)
    )

best_model.fit(X_dev, y_dev)

# Train Baseline Model (Basic features: amt & hour, default threshold 0.5)
baseline_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=4,
)
baseline_model.fit(dev_df[BASELINE_COLS], y_dev)

# =========================================================
# STAGE 5: FINAL EVALUATION ON HELD-OUT TEST SET
# =========================================================
y_pred_base = baseline_model.predict(test_df[BASELINE_COLS])
test_probs_best = best_model.predict_proba(X_test)[:, 1]
y_pred_best = (test_probs_best >= best_threshold).astype(int)

for label, preds in [
    ("BASELINE MODEL (amt + hour, Random Forest)", y_pred_base),
    (
        f"BEST AUTOML MODEL ({best_trial.params['model_family']} | Threshold={best_threshold:.4f})",
        y_pred_best,
    ),
]:
    acc = accuracy_score(y_test, preds)
    f1_mac = f1_score(y_test, preds, average="macro")
    f1_frd = f1_score(y_test, preds, pos_label=1)

    print("\n" + "=" * 60)
    print(label)
    print("=" * 60)
    print(f"Accuracy (Fraud=1, Not Fraud=0) : {acc * 100:.4f}%")
    print(f"Macro-Averaged Overall F1       : {f1_mac:.4f}")
    print(f"F1 Score (Fraud Class Only)     : {f1_frd:.4f}")
    print("-" * 60)
    print(
        classification_report(
            y_test, preds, target_names=["Not Fraud (0)", "Fraud (1)"], digits=4
        )
    )

C:\Users\kiran\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


EXECUTIVE SUMMARY METRICS
Total Transactions : 700,807
Total Volume ($)   : $50,072,503.07
Legitimate Tx      : 697,228 (99.49%)
Fraud Tx           : 3,579 (0.5107%)

Running AutoML search across XGBoost, LightGBM & HistGradientBoosting...

TOP AUTOML CANDIDATES (VALIDATION LEADERBOARD):
 Trial         Model_Family  Val_Fraud_F1  Tuned_Threshold
    12              XGBoost      0.501062         0.982776
    10              XGBoost      0.450254         0.987826
    11              XGBoost      0.382075         0.993620
     5              XGBoost      0.300417         0.994846
    14              XGBoost      0.212202         0.998396
     7              XGBoost      0.193949         0.998152
     8 HistGradientBoosting      0.176430         0.998479
    13              XGBoost      0.161209         0.999365

BASELINE MODEL (amt + hour, Random Forest)
Accuracy (Fraud=1, Not Fraud=0) : 96.5066%
Macro-Averaged Overall F1       : 0.5655
F1 Score (Fraud Class Only)     : 0.1489
-----------

In [3]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "4"

import random
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_recall_curve,
)
from xgboost import XGBClassifier

# ---------------------------------------------------------
# 1. REPRODUCIBILITY & LOAD DATA
# ---------------------------------------------------------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_PATH = "610_ps4_training/trainingT1FD/cct_train.csv"
df = pd.read_csv(DATA_PATH)

df["datetime"] = pd.to_datetime(
    df["trans_date"].astype(str) + " " + df["trans_time"].astype(str)
)
df = df.sort_values("unix_time").reset_index(drop=True)

# ---------------------------------------------------------
# 2. IMPROVED LEAKAGE-FREE FEATURE ENGINEERING
# ---------------------------------------------------------
CATEGORIES = [
    "entertainment",
    "food_dining",
    "gas_transport",
    "grocery_net",
    "grocery_pos",
    "health_fitness",
    "home",
    "kids_pets",
    "misc_net",
    "misc_pos",
    "personal_care",
    "shopping_net",
    "shopping_pos",
    "travel",
]
AGE_GROUPS = ["young_adults", "adults_2550", "adults_50up"]
NO_PREV = 10 * 24 * 3600  # 10 days in seconds


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (
        np.sin((lat2 - lat1) / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2.0) ** 2
    )
    return 6371.0 * 2.0 * np.arcsin(np.sqrt(a))


def build_improved_features(data: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=data.index)
    ts = data["datetime"]
    dob = pd.to_datetime(data["dob"])

    # Per-transaction & demographic features
    out["amt"] = data["amt"].astype(float)
    out["log_amt"] = np.log1p(out["amt"])
    out["hour"] = ts.dt.hour
    out["is_night"] = ((out["hour"] >= 22) | (out["hour"] <= 3)).astype(int)
    out["day_of_week"] = ts.dt.dayofweek
    out["is_weekend"] = (out["day_of_week"] >= 5).astype(int)
    out["age"] = (ts - dob).dt.days / 365.25
    out["gender_male"] = (data["gender"] == "M").astype(int)
    out["log_city_pop"] = np.log1p(data["city_pop"].astype(float))
    out["dist_home_km"] = haversine_km(
        data["lat"], data["long"], data["merch_lat"], data["merch_long"]
    )

    if "profile" in data.columns:
        prof = data["profile"].astype(str)
        out["is_rural"] = prof.str.endswith("_rural").astype(int)
        for g in AGE_GROUPS:
            out[f"agegrp_{g}"] = prof.str.startswith(g).astype(int)

    for c in CATEGORIES:
        out[f"cat_{c}"] = (data["category"] == c).astype(int)

    # Vectorized past-only card burst & spatial jump features
    work = pd.DataFrame({
        "card": data["cc_num"].astype(str).values,
        "t": data["unix_time"].astype(np.int64).values,
        "amt": out["amt"].values,
        "night": out["is_night"].values,
        "cat": data["category"].astype(str).values,
        "mlat": data["merch_lat"].astype(float).values,
        "mlon": data["merch_long"].astype(float).values,
        "pos": np.arange(len(data)),
    }).sort_values(["card", "t", "pos"], kind="mergesort")

    n = len(work)
    card_cols = [
        "secs_since_prev",
        "dist_from_prev_km",
        "implied_speed_kmh",
        "n_txn_1h",
        "n_txn_24h",
        "amt_sum_24h",
        "amt_vs_card_avg",
        "n_night_24h",
        "new_category_for_card",
    ]
    res = {k: np.empty(n) for k in card_cols}

    t_all = work["t"].values
    a_all = work["amt"].values
    ngt_all = work["night"].values
    mlat_all = work["mlat"].values
    mlon_all = work["mlon"].values

    card_vals = work["card"].values
    starts = np.flatnonzero(np.r_[True, card_vals[1:] != card_vals[:-1]])
    ends = np.r_[starts[1:], n]

    for s, e in zip(starts, ends):
        t = t_all[s:e]
        a = a_all[s:e]
        ngt = ngt_all[s:e]
        mlat = mlat_all[s:e]
        mlon = mlon_all[s:e]
        idx = np.arange(e - s)

        # Time since previous transaction on this card
        gap = np.diff(t, prepend=t[0] - NO_PREV).astype(float)
        secs = np.minimum(gap, NO_PREV)
        res["secs_since_prev"][s:e] = secs

        # Geographic jump from previous merchant & implied travel speed
        if e - s > 1:
            jump = np.r_[
                0.0, haversine_km(mlat[:-1], mlon[:-1], mlat[1:], mlon[1:])
            ]
        else:
            jump = np.array([0.0])
        res["dist_from_prev_km"][s:e] = jump
        res["implied_speed_kmh"][s:e] = jump / np.maximum(secs / 3600.0, 1e-3)

        # Rolling 1h and 24h transaction counts & spend
        lo1 = np.searchsorted(t, t - 3600, side="left")
        lo24 = np.searchsorted(t, t - 86400, side="left")
        res["n_txn_1h"][s:e] = idx - lo1 + 1
        res["n_txn_24h"][s:e] = idx - lo24 + 1

        ca = np.r_[0.0, np.cumsum(a)]
        cn = np.r_[0, np.cumsum(ngt)]
        res["amt_sum_24h"][s:e] = ca[idx + 1] - ca[lo24]
        res["n_night_24h"][s:e] = cn[idx + 1] - cn[lo24]

        # Ratio vs card's historical average spend
        past_mean = np.where(idx > 0, ca[idx] / np.maximum(idx, 1), np.nan)
        res["amt_vs_card_avg"][s:e] = np.where(
            idx > 0, a / np.maximum(past_mean, 1.0), 1.0
        )

    res["new_category_for_card"] = (
        (~work.duplicated(["card", "cat"])).astype(int).values
    )

    card_df = pd.DataFrame(res)
    card_df["pos"] = work["pos"].values
    card_df = card_df.sort_values("pos").drop(columns="pos")
    card_df.index = data.index

    return pd.concat([out, card_df], axis=1)


X_all = build_improved_features(df)
y_all = df["is_fraud"].astype(int).values
BASELINE_COLS = ["amt", "hour"]

# ---------------------------------------------------------
# 3. CHRONOLOGICAL SPLIT (DEV vs. HELD-OUT TEST)
# ---------------------------------------------------------
if (df["trans_date"] >= "2022-09-01").sum() > 0:
    dev_mask = (df["trans_date"] < "2022-09-01").values
else:
    dev_mask = np.arange(len(df)) < int(len(df) * 0.80)

X_dev, y_dev = X_all[dev_mask], y_all[dev_mask]
X_test, y_test = X_all[~dev_mask], y_all[~dev_mask]

# Split Dev into 75% Train / 25% Threshold-Tuning Validation
val_split = int(len(X_dev) * 0.75)
X_train, y_train = X_dev.iloc[:val_split], y_dev[:val_split]
X_val, y_val = X_dev.iloc[val_split:], y_dev[val_split:]

# ---------------------------------------------------------
# 4. THRESHOLD TUNING ON VALIDATION SET
# ---------------------------------------------------------
def find_best_threshold(y_true, y_probs):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = np.argmax(f1_scores[:-1])
    return float(thresholds[best_idx]), float(f1_scores[best_idx])


# Balanced weight for XGBoost (equivalent to class_weight="balanced")
train_pos_weight = (y_train == 0).sum() / max((y_train == 1).sum(), 1)

xgb_params = dict(
    n_estimators=300,
    learning_rate=0.08,
    grow_policy="lossguide",
    max_leaves=24,
    max_depth=0,
    subsample=0.85,
    colsample_bytree=0.85,
    tree_method="hist",
    random_state=RANDOM_SEED,
    n_jobs=4,
)

val_xgb = XGBClassifier(**xgb_params, scale_pos_weight=train_pos_weight)
val_xgb.fit(X_train, y_train)
val_probs = val_xgb.predict_proba(X_val)[:, 1]
best_threshold, best_val_f1 = find_best_threshold(y_val, val_probs)
print(
    f"Tuned Validation Threshold: {best_threshold:.4f} (Val Fraud F1: {best_val_f1:.4f})"
)

# ---------------------------------------------------------
# 5. TRAIN FINAL BEST MODEL & BASELINE (BOTH BALANCED)
# ---------------------------------------------------------
# Best Model: XGBoost retrained on all pre-test development rows
dev_pos_weight = (y_dev == 0).sum() / max((y_dev == 1).sum(), 1)
best_model = XGBClassifier(**xgb_params, scale_pos_weight=dev_pos_weight)
best_model.fit(X_dev, y_dev)

# Baseline Model: Random Forest on basic features with class_weight="balanced"
baseline_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=4,
)
baseline_model.fit(X_dev[BASELINE_COLS], y_dev)

# ---------------------------------------------------------
# 6. FINAL COMPARISON ON HELD-OUT TEST SET
# ---------------------------------------------------------
y_pred_base = baseline_model.predict(X_test[BASELINE_COLS])
y_pred_best = (best_model.predict_proba(X_test)[:, 1] >= best_threshold).astype(int)

results_summary = []
for name, preds in [
    ("Baseline (amt + hour, Balanced RF)", y_pred_base),
    (f"Best Model (Improved XGBoost, Balanced, Thresh={best_threshold:.4f})", y_pred_best),
]:
    acc = accuracy_score(y_test, preds)
    f1_mac = f1_score(y_test, preds, average="macro")
    f1_frd = f1_score(y_test, preds, pos_label=1)
    results_summary.append({
        "Model": name,
        "Accuracy (%)": round(acc * 100, 4),
        "Macro F1": round(f1_mac, 4),
        "Fraud F1 (Class 1)": round(f1_frd, 4),
    })

    print("\n" + "=" * 65)
    print(name)
    print("=" * 65)
    print(f"Accuracy (Fraud=1, Not Fraud=0) : {acc * 100:.4f}%")
    print(f"Macro-Averaged Overall F1       : {f1_mac:.4f}")
    print(f"F1 Score (Fraud Class Only)     : {f1_frd:.4f}")
    print("-" * 65)
    print(
        classification_report(
            y_test, preds, target_names=["Not Fraud (0)", "Fraud (1)"], digits=4
        )
    )

print("\nSIDE-BY-SIDE COMPARISON TABLE:")
print(pd.DataFrame(results_summary).to_string(index=False))

Tuned Validation Threshold: 0.9971 (Val Fraud F1: 0.5498)

Baseline (amt + hour, Balanced RF)
Accuracy (Fraud=1, Not Fraud=0) : 96.5066%
Macro-Averaged Overall F1       : 0.5655
F1 Score (Fraud Class Only)     : 0.1489
-----------------------------------------------------------------
               precision    recall  f1-score   support

Not Fraud (0)     0.9994    0.9655    0.9822    139931
    Fraud (1)     0.0816    0.8462    0.1489       507

     accuracy                         0.9651    140438
    macro avg     0.5405    0.9058    0.5655    140438
 weighted avg     0.9961    0.9651    0.9792    140438


Best Model (Improved XGBoost, Balanced, Thresh=0.9971)
Accuracy (Fraud=1, Not Fraud=0) : 99.7465%
Macro-Averaged Overall F1       : 0.7890
F1 Score (Fraud Class Only)     : 0.5792
-----------------------------------------------------------------
               precision    recall  f1-score   support

Not Fraud (0)     0.9981    0.9993    0.9987    139931
    Fraud (1)     0.7227

In [4]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "4"

import random
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_recall_curve,
)
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier

# ---------------------------------------------------------
# 1. REPRODUCIBILITY & LOAD DATA
# ---------------------------------------------------------
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

DATA_PATH = "610_ps4_training/trainingT1FD/cct_train.csv"
df = pd.read_csv(DATA_PATH)

df["datetime"] = pd.to_datetime(
    df["trans_date"].astype(str) + " " + df["trans_time"].astype(str)
)
df = df.sort_values("unix_time").reset_index(drop=True)

# ---------------------------------------------------------
# 2. ENHANCED LEAKAGE-FREE FEATURE ENGINEERING
# ---------------------------------------------------------
CATEGORIES = [
    "entertainment",
    "food_dining",
    "gas_transport",
    "grocery_net",
    "grocery_pos",
    "health_fitness",
    "home",
    "kids_pets",
    "misc_net",
    "misc_pos",
    "personal_care",
    "shopping_net",
    "shopping_pos",
    "travel",
]
AGE_GROUPS = ["young_adults", "adults_2550", "adults_50up"]
NO_PREV = 10 * 24 * 3600  # 10 days in seconds


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (
        np.sin((lat2 - lat1) / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2.0) ** 2
    )
    return 6371.0 * 2.0 * np.arcsin(np.sqrt(a))


def build_enhanced_features(data: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=data.index)
    ts = data["datetime"]
    dob = pd.to_datetime(data["dob"])

    # Per-transaction & demographic features
    out["amt"] = data["amt"].astype(float)
    out["log_amt"] = np.log1p(out["amt"])
    out["hour"] = ts.dt.hour
    out["hour_sin"] = np.sin(2.0 * np.pi * out["hour"] / 24.0)
    out["hour_cos"] = np.cos(2.0 * np.pi * out["hour"] / 24.0)
    out["is_night"] = ((out["hour"] >= 22) | (out["hour"] <= 3)).astype(int)
    out["day_of_week"] = ts.dt.dayofweek
    out["is_weekend"] = (out["day_of_week"] >= 5).astype(int)
    out["age"] = (ts - dob).dt.days / 365.25
    out["gender_male"] = (data["gender"] == "M").astype(int)
    out["log_city_pop"] = np.log1p(data["city_pop"].astype(float))
    out["dist_home_km"] = haversine_km(
        data["lat"], data["long"], data["merch_lat"], data["merch_long"]
    )

    if "profile" in data.columns:
        prof = data["profile"].astype(str)
        out["is_rural"] = prof.str.endswith("_rural").astype(int)
        for g in AGE_GROUPS:
            out[f"agegrp_{g}"] = prof.str.startswith(g).astype(int)

    for c in CATEGORIES:
        out[f"cat_{c}"] = (data["category"] == c).astype(int)

    # Past-only category expanding average spend ratio
    cat_cum_sum = data.groupby("category")["amt"].cumsum() - out["amt"]
    cat_cum_cnt = data.groupby("category").cumcount()
    cat_past_mean = np.where(
        cat_cum_cnt > 0, cat_cum_sum / np.maximum(cat_cum_cnt, 1), out["amt"]
    )
    out["amt_vs_cat_avg"] = out["amt"] / np.maximum(cat_past_mean, 1.0)

    # Past-only merchant frequency (how often this merchant has appeared so far)
    out["merchant_past_tx_count"] = data.groupby("merchant").cumcount()

    # Vectorized past-only card burst, spatial jump & card-merchant novelty features
    work = pd.DataFrame({
        "card": data["cc_num"].astype(str).values,
        "t": data["unix_time"].astype(np.int64).values,
        "amt": out["amt"].values,
        "night": out["is_night"].values,
        "cat": data["category"].astype(str).values,
        "merch": data["merchant"].astype(str).values,
        "mlat": data["merch_lat"].astype(float).values,
        "mlon": data["merch_long"].astype(float).values,
        "pos": np.arange(len(data)),
    }).sort_values(["card", "t", "pos"], kind="mergesort")

    n = len(work)
    card_cols = [
        "secs_since_prev",
        "secs_since_prev_2",
        "dist_from_prev_km",
        "implied_speed_kmh",
        "n_txn_1h",
        "n_txn_24h",
        "amt_sum_24h",
        "amt_vs_card_avg",
        "amt_zscore_card",
        "n_night_24h",
        "new_category_for_card",
        "new_merchant_for_card",
        "card_merchant_past_count",
    ]
    res = {k: np.empty(n) for k in card_cols}

    t_all = work["t"].values
    a_all = work["amt"].values
    ngt_all = work["night"].values
    mlat_all = work["mlat"].values
    mlon_all = work["mlon"].values

    card_vals = work["card"].values
    starts = np.flatnonzero(np.r_[True, card_vals[1:] != card_vals[:-1]])
    ends = np.r_[starts[1:], n]

    for s, e in zip(starts, ends):
        t = t_all[s:e]
        a = a_all[s:e]
        ngt = ngt_all[s:e]
        mlat = mlat_all[s:e]
        mlon = mlon_all[s:e]
        m_len = e - s
        idx = np.arange(m_len)

        # 1st and 2nd order time gaps (detects multi-swipe card testing)
        gap1 = np.diff(t, prepend=t[0] - NO_PREV).astype(float)
        secs1 = np.minimum(gap1, NO_PREV)
        res["secs_since_prev"][s:e] = secs1

        if m_len > 2:
            gap2 = np.r_[NO_PREV, NO_PREV, (t[2:] - t[:-2]).astype(float)]
        else:
            gap2 = np.full(m_len, float(NO_PREV))
        res["secs_since_prev_2"][s:e] = np.minimum(gap2, NO_PREV)

        # Geographic jump from previous merchant & implied travel speed
        if m_len > 1:
            jump = np.r_[
                0.0, haversine_km(mlat[:-1], mlon[:-1], mlat[1:], mlon[1:])
            ]
        else:
            jump = np.array([0.0])
        res["dist_from_prev_km"][s:e] = jump
        res["implied_speed_kmh"][s:e] = jump / np.maximum(secs1 / 3600.0, 1e-3)

        # Rolling 1h and 24h transaction counts & spend
        lo1 = np.searchsorted(t, t - 3600, side="left")
        lo24 = np.searchsorted(t, t - 86400, side="left")
        res["n_txn_1h"][s:e] = idx - lo1 + 1
        res["n_txn_24h"][s:e] = idx - lo24 + 1

        ca = np.r_[0.0, np.cumsum(a)]
        ca2 = np.r_[0.0, np.cumsum(a ** 2)]
        cn = np.r_[0, np.cumsum(ngt)]
        res["amt_sum_24h"][s:e] = ca[idx + 1] - ca[lo24]
        res["n_night_24h"][s:e] = cn[idx + 1] - cn[lo24]

        # Past-only card mean & z-score
        past_mean = np.where(idx > 0, ca[idx] / np.maximum(idx, 1), np.nan)
        past_var = np.where(
            idx > 1,
            np.maximum(
                (ca2[idx] / np.maximum(idx, 1)) - (past_mean ** 2), 1.0
            ),
            100.0,
        )
        res["amt_vs_card_avg"][s:e] = np.where(
            idx > 0, a / np.maximum(past_mean, 1.0), 1.0
        )
        res["amt_zscore_card"][s:e] = np.where(
            idx > 1, (a - past_mean) / np.sqrt(past_var), 0.0
        )

    # First-time category & merchant flags + repeat count for this card
    res["new_category_for_card"] = (
        (~work.duplicated(["card", "cat"])).astype(int).values
    )
    res["new_merchant_for_card"] = (
        (~work.duplicated(["card", "merch"])).astype(int).values
    )
    res["card_merchant_past_count"] = (
        work.groupby(["card", "merch"]).cumcount().values
    )

    card_df = pd.DataFrame(res)
    card_df["pos"] = work["pos"].values
    card_df = card_df.sort_values("pos").drop(columns="pos")
    card_df.index = data.index

    return pd.concat([out, card_df], axis=1)


X_all = build_enhanced_features(df)
y_all = df["is_fraud"].astype(int).values
BASELINE_COLS = ["amt", "hour"]

# ---------------------------------------------------------
# 3. CHRONOLOGICAL SPLIT (DEV vs. HELD-OUT TEST)
# ---------------------------------------------------------
if (df["trans_date"] >= "2022-09-01").sum() > 0:
    dev_mask = (df["trans_date"] < "2022-09-01").values
else:
    dev_mask = np.arange(len(df)) < int(len(df) * 0.80)

X_dev, y_dev = X_all[dev_mask], y_all[dev_mask]
X_test, y_test = X_all[~dev_mask], y_all[~dev_mask]

# ---------------------------------------------------------
# 4. 3 EXPANDING TIME WINDOWS FOR THRESHOLD TUNING
# ---------------------------------------------------------
def find_best_threshold(y_true, y_probs):
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    f1_scores = (2 * precisions * recalls) / (precisions + recalls + 1e-12)
    best_idx = np.argmax(f1_scores[:-1])
    return float(thresholds[best_idx]), float(f1_scores[best_idx])


xgb_params = dict(
    n_estimators=350,
    learning_rate=0.07,
    grow_policy="lossguide",
    max_leaves=31,
    max_depth=0,
    min_child_weight=5,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_alpha=1.0,
    reg_lambda=2.0,
    tree_method="hist",
    random_state=RANDOM_SEED,
    n_jobs=4,
)

# Pool out-of-fold probabilities across 3 expanding chronological windows
n_dev = len(X_dev)
window_edges = [
    int(n_dev * 0.55),
    int(n_dev * 0.70),
    int(n_dev * 0.85),
    n_dev,
]

oof_probs = []
oof_targets = []

for w in range(3):
    tr_end = window_edges[w]
    va_end = window_edges[w + 1]

    X_w_tr, y_w_tr = X_dev.iloc[:tr_end], y_dev[:tr_end]
    X_w_va, y_w_va = X_dev.iloc[tr_end:va_end], y_dev[tr_end:va_end]

    # Exact class_weight="balanced" sample weights
    w_sample_weights = compute_sample_weight("balanced", y_w_tr)

    win_model = XGBClassifier(**xgb_params)
    win_model.fit(X_w_tr, y_w_tr, sample_weight=w_sample_weights)

    oof_probs.append(win_model.predict_proba(X_w_va)[:, 1])
    oof_targets.append(y_w_va)

pooled_probs = np.concatenate(oof_probs)
pooled_targets = np.concatenate(oof_targets)
best_threshold, pooled_val_f1 = find_best_threshold(pooled_targets, pooled_probs)
print(
    f"3-Window Pooled Threshold: {best_threshold:.4f} (Pooled Dev Fraud F1: {pooled_val_f1:.4f})"
)

# ---------------------------------------------------------
# 5. TRAIN FINAL BEST MODEL & BASELINE (BOTH BALANCED)
# ---------------------------------------------------------
# Best Model: XGBoost retrained on all pre-test development rows with balanced weights
dev_sample_weights = compute_sample_weight("balanced", y_dev)
best_model = XGBClassifier(**xgb_params)
best_model.fit(X_dev, y_dev, sample_weight=dev_sample_weights)

# Baseline Model: Random Forest on basic features with class_weight="balanced"
baseline_model = RandomForestClassifier(
    n_estimators=100,
    max_depth=12,
    class_weight="balanced",
    random_state=RANDOM_SEED,
    n_jobs=4,
)
baseline_model.fit(X_dev[BASELINE_COLS], y_dev)

# ---------------------------------------------------------
# 6. FINAL SIDE-BY-SIDE COMPARISON ON HELD-OUT TEST SET
# ---------------------------------------------------------
y_pred_base = baseline_model.predict(X_test[BASELINE_COLS])
y_pred_best = (best_model.predict_proba(X_test)[:, 1] >= best_threshold).astype(int)

results_summary = []
for name, preds in [
    ("Baseline (amt + hour, Balanced RF)", y_pred_base),
    (
        f"Best Model (Enhanced XGBoost, Balanced, Thresh={best_threshold:.4f})",
        y_pred_best,
    ),
]:
    acc = accuracy_score(y_test, preds)
    f1_mac = f1_score(y_test, preds, average="macro")
    f1_frd = f1_score(y_test, preds, pos_label=1)
    results_summary.append({
        "Model": name,
        "Accuracy (%)": round(acc * 100, 4),
        "Macro F1": round(f1_mac, 4),
        "Fraud F1 (Class 1)": round(f1_frd, 4),
    })

    print("\n" + "=" * 68)
    print(name)
    print("=" * 68)
    print(f"Accuracy (Fraud=1, Not Fraud=0) : {acc * 100:.4f}%")
    print(f"Macro-Averaged Overall F1       : {f1_mac:.4f}")
    print(f"F1 Score (Fraud Class Only)     : {f1_frd:.4f}")
    print("-" * 68)
    print(
        classification_report(
            y_test, preds, target_names=["Not Fraud (0)", "Fraud (1)"], digits=4
        )
    )

print("\nSIDE-BY-SIDE COMPARISON TABLE:")
print(pd.DataFrame(results_summary).to_string(index=False))

3-Window Pooled Threshold: 0.9474 (Pooled Dev Fraud F1: 0.8522)

Baseline (amt + hour, Balanced RF)
Accuracy (Fraud=1, Not Fraud=0) : 96.5066%
Macro-Averaged Overall F1       : 0.5655
F1 Score (Fraud Class Only)     : 0.1489
--------------------------------------------------------------------
               precision    recall  f1-score   support

Not Fraud (0)     0.9994    0.9655    0.9822    139931
    Fraud (1)     0.0816    0.8462    0.1489       507

     accuracy                         0.9651    140438
    macro avg     0.5405    0.9058    0.5655    140438
 weighted avg     0.9961    0.9651    0.9792    140438


Best Model (Enhanced XGBoost, Balanced, Thresh=0.9474)
Accuracy (Fraud=1, Not Fraud=0) : 99.9445%
Macro-Averaged Overall F1       : 0.9589
F1 Score (Fraud Class Only)     : 0.9181
--------------------------------------------------------------------
               precision    recall  f1-score   support

Not Fraud (0)     0.9995    0.9999    0.9997    139931
    Fraud (1

In [5]:
import pickle

model_bundle = {
    "model": best_model,
    "threshold": float(best_threshold),
    "feature_cols": list(X_dev.columns),
}

with open("xgboost_best_model.pkl", "wb") as f:
    pickle.dump(model_bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

print(f"Saved xgboost_best_model.pkl with threshold={best_threshold:.4f} and {len(X_dev.columns)} features.")

Saved xgboost_best_model.pkl with threshold=0.9474 and 45 features.


# Testing on Synthetic Dataset

In [6]:
import os

os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"
os.environ["OMP_NUM_THREADS"] = "4"

import pickle
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report, f1_score

# ---------------------------------------------------------
# 1. EXACT FEATURE ENGINEERING FUNCTION
# ---------------------------------------------------------
CATEGORIES = [
    "entertainment",
    "food_dining",
    "gas_transport",
    "grocery_net",
    "grocery_pos",
    "health_fitness",
    "home",
    "kids_pets",
    "misc_net",
    "misc_pos",
    "personal_care",
    "shopping_net",
    "shopping_pos",
    "travel",
]
AGE_GROUPS = ["young_adults", "adults_2550", "adults_50up"]
NO_PREV = 10 * 24 * 3600  # 10 days in seconds


def haversine_km(lat1, lon1, lat2, lon2):
    lat1, lon1, lat2, lon2 = map(np.radians, (lat1, lon1, lat2, lon2))
    a = (
        np.sin((lat2 - lat1) / 2.0) ** 2
        + np.cos(lat1) * np.cos(lat2) * np.sin((lon2 - lon1) / 2.0) ** 2
    )
    return 6371.0 * 2.0 * np.arcsin(np.sqrt(a))


def build_enhanced_features(data: pd.DataFrame) -> pd.DataFrame:
    out = pd.DataFrame(index=data.index)
    ts = pd.to_datetime(
        data["trans_date"].astype(str) + " " + data["trans_time"].astype(str)
    )
    dob = pd.to_datetime(data["dob"])

    # Per-transaction & demographic features
    out["amt"] = data["amt"].astype(float)
    out["log_amt"] = np.log1p(out["amt"])
    out["hour"] = ts.dt.hour
    out["hour_sin"] = np.sin(2.0 * np.pi * out["hour"] / 24.0)
    out["hour_cos"] = np.cos(2.0 * np.pi * out["hour"] / 24.0)
    out["is_night"] = ((out["hour"] >= 22) | (out["hour"] <= 3)).astype(int)
    out["day_of_week"] = ts.dt.dayofweek
    out["is_weekend"] = (out["day_of_week"] >= 5).astype(int)
    out["age"] = (ts - dob).dt.days / 365.25
    out["gender_male"] = (data["gender"] == "M").astype(int)
    out["log_city_pop"] = np.log1p(data["city_pop"].astype(float))
    out["dist_home_km"] = haversine_km(
        data["lat"], data["long"], data["merch_lat"], data["merch_long"]
    )

    if "profile" in data.columns:
        prof = data["profile"].astype(str)
        out["is_rural"] = prof.str.endswith("_rural").astype(int)
        for g in AGE_GROUPS:
            out[f"agegrp_{g}"] = prof.str.startswith(g).astype(int)

    for c in CATEGORIES:
        out[f"cat_{c}"] = (data["category"] == c).astype(int)

    # Past-only category expanding average spend ratio
    cat_cum_sum = data.groupby("category")["amt"].cumsum() - out["amt"]
    cat_cum_cnt = data.groupby("category").cumcount()
    cat_past_mean = np.where(
        cat_cum_cnt > 0, cat_cum_sum / np.maximum(cat_cum_cnt, 1), out["amt"]
    )
    out["amt_vs_cat_avg"] = out["amt"] / np.maximum(cat_past_mean, 1.0)

    # Past-only merchant frequency
    out["merchant_past_tx_count"] = data.groupby("merchant").cumcount()

    # Vectorized past-only card burst, spatial jump & card-merchant novelty features
    work = pd.DataFrame({
        "card": data["cc_num"].astype(str).values,
        "t": data["unix_time"].astype(np.int64).values,
        "amt": out["amt"].values,
        "night": out["is_night"].values,
        "cat": data["category"].astype(str).values,
        "merch": data["merchant"].astype(str).values,
        "mlat": data["merch_lat"].astype(float).values,
        "mlon": data["merch_long"].astype(float).values,
        "pos": np.arange(len(data)),
    }).sort_values(["card", "t", "pos"], kind="mergesort")

    n = len(work)
    card_cols = [
        "secs_since_prev",
        "secs_since_prev_2",
        "dist_from_prev_km",
        "implied_speed_kmh",
        "n_txn_1h",
        "n_txn_24h",
        "amt_sum_24h",
        "amt_vs_card_avg",
        "amt_zscore_card",
        "n_night_24h",
        "new_category_for_card",
        "new_merchant_for_card",
        "card_merchant_past_count",
    ]
    res = {k: np.empty(n) for k in card_cols}

    t_all = work["t"].values
    a_all = work["amt"].values
    ngt_all = work["night"].values
    mlat_all = work["mlat"].values
    mlon_all = work["mlon"].values

    card_vals = work["card"].values
    starts = np.flatnonzero(np.r_[True, card_vals[1:] != card_vals[:-1]])
    ends = np.r_[starts[1:], n]

    for s, e in zip(starts, ends):
        t = t_all[s:e]
        a = a_all[s:e]
        ngt = ngt_all[s:e]
        mlat = mlat_all[s:e]
        mlon = mlon_all[s:e]
        m_len = e - s
        idx = np.arange(m_len)

        gap1 = np.diff(t, prepend=t[0] - NO_PREV).astype(float)
        secs1 = np.minimum(gap1, NO_PREV)
        res["secs_since_prev"][s:e] = secs1

        if m_len > 2:
            gap2 = np.r_[NO_PREV, NO_PREV, (t[2:] - t[:-2]).astype(float)]
        else:
            gap2 = np.full(m_len, float(NO_PREV))
        res["secs_since_prev_2"][s:e] = np.minimum(gap2, NO_PREV)

        if m_len > 1:
            jump = np.r_[
                0.0, haversine_km(mlat[:-1], mlon[:-1], mlat[1:], mlon[1:])
            ]
        else:
            jump = np.array([0.0])
        res["dist_from_prev_km"][s:e] = jump
        res["implied_speed_kmh"][s:e] = jump / np.maximum(secs1 / 3600.0, 1e-3)

        lo1 = np.searchsorted(t, t - 3600, side="left")
        lo24 = np.searchsorted(t, t - 86400, side="left")
        res["n_txn_1h"][s:e] = idx - lo1 + 1
        res["n_txn_24h"][s:e] = idx - lo24 + 1

        ca = np.r_[0.0, np.cumsum(a)]
        ca2 = np.r_[0.0, np.cumsum(a ** 2)]
        cn = np.r_[0, np.cumsum(ngt)]
        res["amt_sum_24h"][s:e] = ca[idx + 1] - ca[lo24]
        res["n_night_24h"][s:e] = cn[idx + 1] - cn[lo24]

        past_mean = np.where(idx > 0, ca[idx] / np.maximum(idx, 1), np.nan)
        past_var = np.where(
            idx > 1,
            np.maximum(
                (ca2[idx] / np.maximum(idx, 1)) - (past_mean ** 2), 1.0
            ),
            100.0,
        )
        res["amt_vs_card_avg"][s:e] = np.where(
            idx > 0, a / np.maximum(past_mean, 1.0), 1.0
        )
        res["amt_zscore_card"][s:e] = np.where(
            idx > 1, (a - past_mean) / np.sqrt(past_var), 0.0
        )

    res["new_category_for_card"] = (
        (~work.duplicated(["card", "cat"])).astype(int).values
    )
    res["new_merchant_for_card"] = (
        (~work.duplicated(["card", "merch"])).astype(int).values
    )
    res["card_merchant_past_count"] = (
        work.groupby(["card", "merch"]).cumcount().values
    )

    card_df = pd.DataFrame(res)
    card_df["pos"] = work["pos"].values
    card_df = card_df.sort_values("pos").drop(columns="pos")
    card_df.index = data.index

    return pd.concat([out, card_df], axis=1)


# ---------------------------------------------------------
# 2. LOAD SYNTHETIC TEST DATA & SAVED PICKLE MODEL
# ---------------------------------------------------------
test_df = pd.read_csv("synthetic_test_100.csv")
test_df = test_df.sort_values("unix_time").reset_index(drop=True)

with open("xgboost_best_model.pkl", "rb") as f:
    bundle = pickle.load(f)

loaded_model = bundle["model"]
threshold = bundle["threshold"]
feature_cols = bundle["feature_cols"]

# ---------------------------------------------------------
# 3. BUILD FEATURES & RUN PREDICTIONS
# ---------------------------------------------------------
X_test_synth = build_enhanced_features(test_df)

# Align columns safely in case any optional column (like 'profile') differs
for col in feature_cols:
    if col not in X_test_synth.columns:
        X_test_synth[col] = 0
X_test_synth = X_test_synth[feature_cols]

fraud_probs = loaded_model.predict_proba(X_test_synth)[:, 1]
y_pred = (fraud_probs >= threshold).astype(int)

# ---------------------------------------------------------
# 4. DISPLAY PREDICTIONS & EVALUATION METRICS
# ---------------------------------------------------------
results_df = test_df[
    ["trans_num", "trans_date", "trans_time", "category", "amt"]
].copy()
results_df["fraud_prob"] = np.round(fraud_probs, 4)
results_df["predicted_fraud"] = y_pred

if "is_fraud" in test_df.columns:
    y_true = test_df["is_fraud"].astype(int).values
    results_df["actual_fraud"] = y_true

    print("=" * 60)
    print("TEST EVALUATION ON synthetic_test_100.csv")
    print("=" * 60)
    print(f"Tuned Threshold Used            : {threshold:.4f}")
    print(f"Predicted Fraud Count           : {y_pred.sum()} / {len(y_pred)}")
    print(
        f"Accuracy (Fraud=1, Not Fraud=0) : {accuracy_score(y_true, y_pred) * 100:.2f}%"
    )
    print(
        f"Macro-Averaged Overall F1       : {f1_score(y_true, y_pred, average='macro'):.4f}"
    )
    print(
        f"F1 Score (Fraud Class Only)     : {f1_score(y_true, y_pred, pos_label=1):.4f}"
    )
    print("-" * 60)
    print(
        classification_report(
            y_true, y_pred, target_names=["Not Fraud (0)", "Fraud (1)"], digits=4
        )
    )
else:
    print(f"Predicted Fraud Count: {y_pred.sum()} / {len(y_pred)}")

print("\nTop 15 Transactions by Predicted Fraud Probability:")
display(results_df.sort_values("fraud_prob", ascending=False).head(15))

TEST EVALUATION ON synthetic_test_100.csv
Tuned Threshold Used            : 0.9474
Predicted Fraud Count           : 31 / 100
Accuracy (Fraud=1, Not Fraud=0) : 84.00%
Macro-Averaged Overall F1       : 0.7867
F1 Score (Fraud Class Only)     : 0.6800
------------------------------------------------------------
               precision    recall  f1-score   support

Not Fraud (0)     0.9710    0.8272    0.8933        81
    Fraud (1)     0.5484    0.8947    0.6800        19

     accuracy                         0.8400       100
    macro avg     0.7597    0.8609    0.7867       100
 weighted avg     0.8907    0.8400    0.8528       100


Top 15 Transactions by Predicted Fraud Probability:


,trans_num,trans_date,trans_time,category,amt,fraud_prob,predicted_fraud,actual_fraud
26,tx_test_0030,2022-09-09,02:17:43,misc_net,1019.47,1.0000,1,1
17,tx_test_0016,2022-09-06,03:38:13,grocery_pos,805.47,0.9997,1,1
93,tx_test_0093,2022-09-28,22:40:08,grocery_pos,1183.37,0.9997,1,1
4,tx_test_0002,2022-09-02,23:28:45,grocery_pos,569.51,0.9996,1,1
73,tx_test_0071,2022-09-24,23:48:30,shopping_net,1361.37,0.9995,1,1
16,tx_test_0021,2022-09-06,02:01:56,shopping_net,396.88,0.9995,1,1
58,tx_test_0054,2022-09-18,22:59:56,misc_net,705.80,0.9994,1,1
6,tx_test_0006,2022-09-04,00:59:11,shopping_net,1228.12,0.9993,1,1
32,tx_test_0032,2022-09-10,15:27:30,health_fitness,130.59,0.9987,1,0
30,tx_test_0027,2022-09-09,23:43:21,misc_net,438.32,0.9985,1,1
